In [1]:
from typing import List, Dict, Set, Tuple
import pandas as pd
from tqdm import tqdm
from simpletransformers.ner import NERModel

c:\Users\sward\.conda\envs\adfler\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def process_predictions(predictions: List[List[Dict]]) -> List[List[str]]:
    """Convert SimpleTransformers prediction format to list of labels."""
    processed_preds = []
    for sentence in predictions:
        # Each sentence is a list of dictionaries with one item
        labels = [list(word_dict.values())[0] for word_dict in sentence]
        processed_preds.append(labels)
    return processed_preds

In [4]:
def extract_spans(prediction, text=None):
    """
    Extract valid and invalid spans from NER predictions following BIOES tagging scheme.
    
    Args:
        prediction (list): List of dictionaries, each containing a token and its BIOES tag
        text (str, optional): Original text input (not used in this implementation but included for future extensions)
    
    Returns:
        dict: Dictionary with two keys:
            - 'valid_spans': List of valid entity spans
            - 'invalid_spans': List of invalid entity spans
    """
    valid_spans = []
    invalid_spans = []
    
    current_span = []
    current_entity_type = None
    
    # Helper function to check if a tag follows BIOES scheme
    def is_valid_bioes_transition(prev_tag, current_tag):
        if prev_tag is None:
            return current_tag.startswith('B-') or current_tag.startswith('S-')
        
        prev_prefix = prev_tag[0]
        prev_entity = prev_tag[2:] if len(prev_tag) > 2 else ''
        current_prefix = current_tag[0]
        current_entity = current_tag[2:] if len(current_tag) > 2 else ''
        
        # Entity types must match within a span
        if prev_entity != current_entity and prev_prefix in ['B', 'I'] and current_prefix in ['I', 'E']:
            return False
            
        # Valid transitions
        if prev_prefix == 'B':
            return (current_prefix == 'I' or current_prefix == 'E')
        elif prev_prefix == 'I':
            return (current_prefix == 'I' or current_prefix == 'E')
        elif prev_prefix == 'E' or prev_prefix == 'S':
            return (current_prefix == 'B' or current_prefix == 'S')
        
        return False
    
    prev_tag = None
    for token_dict in prediction:
        token = list(token_dict.keys())[0]
        tag = token_dict[token]
        
        # Process the tag
        if tag.startswith('B-'):  # Beginning of a span
            # If we were building a span, finalize it
            if current_span:
                # Check if the previous span ended properly (with E- or S-)
                if not (prev_tag.startswith('E-') or prev_tag.startswith('S-')):
                    invalid_spans.append((current_span, current_entity_type))
                else:
                    valid_spans.append((current_span, current_entity_type))
                current_span = []
                
            current_span.append(token)
            current_entity_type = tag[2:]  # Extract entity type (after 'B-')
        
        elif tag.startswith('I-'):  # Inside of a span
            # Must follow B- or I- of same entity type
            if (prev_tag and (prev_tag.startswith('B-') or prev_tag.startswith('I-')) and 
                prev_tag[2:] == tag[2:]):
                current_span.append(token)
            else:
                # Invalid: I- tag not following proper B-/I- tag
                if current_span:
                    invalid_spans.append((current_span, current_entity_type))
                current_span = [token]
                current_entity_type = tag[2:]
        
        elif tag.startswith('E-'):  # End of a span
            # Must follow B- or I- of same entity type
            if (prev_tag and (prev_tag.startswith('B-') or prev_tag.startswith('I-')) and 
                prev_tag[2:] == tag[2:]):
                current_span.append(token)
                valid_spans.append((current_span, current_entity_type))
                current_span = []
                current_entity_type = None
            else:
                # Invalid: E- tag not following proper B-/I- tag
                if current_span:
                    invalid_spans.append((current_span, current_entity_type))
                invalid_spans.append(([token], tag[2:]))
                current_span = []
                current_entity_type = None
        
        elif tag.startswith('S-'):  # Single token span
            # If we were building a span, finalize it (as invalid)
            if current_span:
                invalid_spans.append((current_span, current_entity_type))
                current_span = []
            
            # Add this as a single-token valid span
            valid_spans.append(([token], tag[2:]))
        
        elif tag == 'O':  # Outside any span
            # If we were building a span, finalize it (as invalid)
            if current_span:
                invalid_spans.append((current_span, current_entity_type))
                current_span = []
                current_entity_type = None
        
        else:
            # Unknown tag format
            if current_span:
                invalid_spans.append((current_span, current_entity_type))
                current_span = []
            invalid_spans.append(([token], "Unknown"))
        
        prev_tag = tag
    
    # Handle any remaining span
    if current_span:
        # Check if it's a valid span (must end with E-)
        if prev_tag and prev_tag.startswith('E-'):
            valid_spans.append((current_span, current_entity_type))
        else:
            invalid_spans.append((current_span, current_entity_type))
    
    # Format the results
    formatted_valid_spans = []
    for span_tokens, entity_type in valid_spans:
        formatted_valid_spans.append({
            'text': ' '.join(span_tokens),
            'tokens': span_tokens,
            'entity_type': entity_type
        })
    
    formatted_invalid_spans = []
    for span_tokens, entity_type in invalid_spans:
        formatted_invalid_spans.append({
            'text': ' '.join(span_tokens),
            'tokens': span_tokens,
            'entity_type': entity_type,
        })
    
    return {
        'valid_spans': formatted_valid_spans,
        'invalid_spans': formatted_invalid_spans
    }


In [5]:
import os
import torch
import pysbd
import datetime

labels = ['O',
                'B-Event', 'I-Event', 'E-Event', 'S-Event',
                'B-NonEvent', 'I-NonEvent', 'E-NonEvent', 'S-NonEvent',
                ]
use_cuda = True if torch.cuda.is_available() == True else False
droner = NERModel(
            "xlnet",
            "ADFLER-xlnet-base-cased",
            labels=labels,
            use_cuda=use_cuda
        )

segmenter = pysbd.Segmenter(clean=True)
mode = 'semantic'
log_files = os.listdir('data')
for file in log_files:
    filename = file.split('.')[0].split('_')[1:]
    file_path = os.path.join('data', file)
    dataframe = pd.read_excel(file_path)
    # dataframe.to_csv(os.path.join('data',f"{('_').join(filename)}.csv"), index=False)
    timeline = pd.read_excel(file_path)
    print("Forensic timeline is loaded successfully\n")
    print('Start recognizing mentioned entities...')
    pred_list = []
    start = datetime.datetime.now()
    print(f'start: {start}')
    for row in tqdm(range(0, timeline.shape[0])):
        message = timeline.iloc[row, -1]
        if mode == 'semantic':
            predictions, _ = droner.predict([message])
            entities = predictions[0]
            result = extract_spans(predictions[0])
            
            print("Valid spans:")
            for span in result['valid_spans']:
                print(f"- {span['text']} ({span['entity_type']})")
            
            print("\nInvalid spans:")
            for span in result['invalid_spans']:
                print(f"- {span['text']} ({span['entity_type']})")
        else:
            print("Valid spans:")
            print(segmenter.segment(message))
    print(f'end: {datetime.datetime.now()}')
        # pred_labels = process_predictions(predictions)
        # print(message)
        # print(entities)
        # print(pred_labels)
            # timestamp = timeline.iloc[row, 0]
            # pred_list.append({"timestamp": str(timestamp), "entities": entities})

Forensic timeline is loaded successfully

Start recognizing mentioned entities...
start: 2025-08-17 15:55:30.341075


  5%|▍         | 1/22 [00:14<05:10, 14.80s/it]

Valid spans:
- Motors starting.; (Event)
- Flight mode changed to P-GPS. (Event)

Invalid spans:


  9%|▉         | 2/22 [00:26<04:25, 13.26s/it]

Valid spans:
- Data Recorder File Index is 80.; (Event)
- Set Return to Home (RTH) altitude to 100 (NonEvent)
- (328 ft).; (NonEvent)
- Set maximum flight altitude to 500 (NonEvent)
- (1640 ft).; (NonEvent)
- Flight mode changed to Starting Motors. (Event)

Invalid spans:
- m (NonEvent)
- m (NonEvent)


...

Valid spans:
- Image transmission signal weak. (Event)

Invalid spans:
- Adjust antennas and make sure they are perpendicular to flight direction of aircraft. (NonEvent)


 73%|███████▎  | 16/22 [02:01<00:40,  6.81s/it]

Valid spans:
- Image transmission signal weak. (Event)

Invalid spans:


 73%|███████▎  | 16/22 [02:08<00:48,  8.03s/it]


KeyboardInterrupt: 